# <font color="#418FDE" size="6.5" uppercase>**Expansion Binning**</font>

>Last update: 20260712.
    
By the end of this Lecture, you will be able to:
- Use binarization and discretization to convert continuous features into thresholded or binned representations. 
- Generate polynomial, interaction, and spline features for nonlinear modeling. 
- Build leakage-safe preprocessing steps using transformers inside pipelines. 


## **1. Thresholds and Bins**

### **1.1. Threshold Feature Flags**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_B/image_01_01.jpg?v=1783906471" width="250">



>* Flags mark meaningful numerical cutoffs
>* Useful when outcomes change sharply

>* Binary flags reveal meaningful threshold states
>* They aid modeling and stakeholder interpretation

>* Choose cutoffs carefully to avoid misleading signals
>* Use flags alongside original continuous features



In [ ]:
#@title Python Code - Threshold Feature Flags

# Threshold flags mark meaningful numeric cutoffs.
# This example uses simple built-in tools.
# We keep outputs small and readable.

import pandas as pd
import matplotlib.pyplot as plt

# Create a tiny delivery dataset.
deliveries = pd.DataFrame(
    {
        "package": ["A", "B", "C", "D", "E", "F"],
        "delay_hours": [0.5, 1.2, 3.8, 4.1, 7.5, 9.0],
    }
)

# Choose a business threshold.
late_threshold_hours = 4.0

# Add a threshold feature flag.
deliveries["late_flag"] = (
    deliveries["delay_hours"] >= late_threshold_hours
).astype(int)

# Show only compact teaching output.
print("Threshold feature flag example")
print("Late threshold:", late_threshold_hours, "hours")
print(deliveries.to_string(index=False))

# Count packages on each side.
flag_counts = deliveries["late_flag"].value_counts().sort_index()
print("Not late count:", int(flag_counts.get(0, 0)))
print("Late count:", int(flag_counts.get(1, 0)))

# Plot raw values and threshold.
plt.figure(figsize=(6, 3))
plt.bar(deliveries["package"], deliveries["delay_hours"])
plt.axhline(late_threshold_hours, color="red", linestyle="--")
plt.title("Delay Hours With Late Threshold")
plt.xlabel("Package")
plt.ylabel("Delay Hours")
plt.tight_layout()
plt.show()



### **1.2. Discretizing Continuous Features**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_B/image_01_02.jpg?v=1783906473" width="250">



>* Convert numeric values into meaningful bins
>* Capture sharp patterns more easily

>* Bins reduce noise and improve interpretability
>* Broad bins can hide useful detail

>* Choose bin boundaries carefully for modeling.
>* Fit bins on training data only.



In [ ]:
#@title Python Code - Discretizing Continuous Features

# Discretization turns measurements into useful interval labels.
# This example uses only pandas and numpy.
# Bins must be learned from training data.

import numpy as np
import pandas as pd

# Create a tiny continuous feature dataset.
incomes = np.array([28, 35, 42, 55, 61, 73, 88, 120])

# Store values in a small dataframe.
data = pd.DataFrame({"income_k": incomes})

# Define readable income bin boundaries.
bin_edges = [0, 50, 80, np.inf]
bin_names = ["low", "middle", "high"]

# Convert continuous income into categories.
data["income_bin"] = pd.cut(
    data["income_k"], bin_edges, labels=bin_names
)

# Create equal frequency bins from training values.
data["quantile_bin"] = pd.qcut(
    data["income_k"], q=3, labels=["Q1", "Q2", "Q3"]
)

# Show only a compact teaching table.
print("Original income and discretized versions:")
print(data.head(8).to_string(index=False))

# Summarize how many observations entered each bin.
counts = data["income_bin"].value_counts(sort=False)
print("\nCounts per domain bin:")
print(counts.to_string())



### **1.3. Binning Strategies**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_B/image_01_03.jpg?v=1783906475" width="250">



>* Group continuous values into meaningful ranges
>* Use bins carefully; they lose detail

>* Equal-width bins split ranges evenly.
>* Skewed data can create sparse bins.

>* Equal-frequency bins balance skewed data groups
>* Use domain rules and training data only



In [ ]:
#@title Python Code - Binning Strategies

# Compare common binning strategies.
# Use tiny purchase amount data.
# Keep preprocessing training-data only.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create skewed purchase amounts.
purchases = np.array([
    8, 12, 15, 18,
    22, 25, 28, 35,
    45, 60, 90, 180
])

# Validate the tiny example data.
assert purchases.ndim == 1
assert purchases.size == 12

# Split before learning bin boundaries.
train_values = purchases[:8]
test_values = purchases[8:]

# Learn equal-width edges from training data.
width_edges = np.linspace(
    train_values.min(), train_values.max(), 4
)

# Learn equal-frequency edges from training data.
quantile_edges = np.quantile(
    train_values, [0, 1/3, 2/3, 1]
)

# Define domain-informed spending edges.
domain_edges = np.array([0, 20, 50, 200])

# Assign bins using learned boundaries.
width_bins = pd.cut(purchases, width_edges, include_lowest=True)
quantile_bins = pd.cut(purchases, quantile_edges, include_lowest=True)

# Assign bins using expert boundaries.
domain_bins = pd.cut(purchases, domain_edges, include_lowest=True)

# Count observations in each strategy.
width_counts = pd.Series(width_bins).value_counts(sort=False)
quantile_counts = pd.Series(quantile_bins).value_counts(sort=False)
domain_counts = pd.Series(domain_bins).value_counts(sort=False)

# Print compact teaching output.
print("Training values define learned bin edges only.")
print("Equal-width edges:", np.round(width_edges, 1).tolist())
print("Equal-frequency edges:", np.round(quantile_edges, 1).tolist())
print("Domain-informed edges:", domain_edges.tolist())
print("Equal-width counts:", width_counts.tolist())
print("Equal-frequency counts:", quantile_counts.tolist())
print("Domain-informed counts:", domain_counts.tolist())

# Plot the three bin count patterns.
fig, ax = plt.subplots(figsize=(7, 4))
labels = ["Equal width", "Equal frequency", "Domain informed"]
counts = [width_counts.tolist(), quantile_counts.tolist(), domain_counts.tolist()]

# Draw grouped bars for comparison.
x_positions = np.arange(3)
for index, values in enumerate(counts):
    ax.bar(x_positions + index * 0.25, values, width=0.25)

# Label the compact comparison plot.
ax.set_xticks(x_positions + 0.25)
ax.set_xticklabels(["Bin 1", "Bin 2", "Bin 3"])
ax.set_ylabel("Number of purchases")
ax.legend(labels)

# Show the single required plot.
plt.tight_layout()
plt.show()



## **2. Feature Expansion**

### **2.1. Polynomial Feature Expansion**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_B/image_02_01.jpg?v=1783906467" width="250">



>* Adds curved patterns to linear models
>* Creates transformed features for nonlinear relationships

>* Add squared and cubed feature versions
>* Help linear models capture curved patterns

>* Avoid overfitting with excessive polynomial terms
>* Use scaling, validation, and regularization



In [ ]:
#@title Python Code - Polynomial Feature Expansion

# Demonstrate polynomial feature expansion clearly.
# Use tiny arrays for fast execution.
# Compare linear and curved feature columns.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create one continuous feature in meters.
x_meters = np.array([-3, -2, -1, 0, 1, 2, 3], dtype=float)

# Validate the feature before expansion.
assert x_meters.ndim == 1 and x_meters.size == 7

# Build polynomial columns manually.
expanded = pd.DataFrame({
    "distance_m": x_meters,
    "distance_squared": x_meters ** 2,
    "distance_cubed": x_meters ** 3,
})

# Show only a compact preview.
print("Polynomial expansion adds curved feature columns.")
print(expanded.head(4).to_string(index=False))

# Create a simple curved target.
y_value = 2 + 0.5 * x_meters + 1.2 * x_meters ** 2

# Plot original feature against curved target.
plt.figure(figsize=(6, 4))
plt.scatter(x_meters, y_value, color="navy", label="observed pattern")
plt.plot(x_meters, y_value, color="orange", label="curved relationship")
plt.xlabel("Distance from center, meters")
plt.ylabel("Target value")
plt.title("Polynomial terms help represent curves")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



### **2.2. Interaction Features**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_B/image_02_02.jpg?v=1783906465" width="250">



>* Combine features to capture joint effects
>* Model flexible patterns beyond simple addition

>* Combine features into meaningful new measurements
>* Reveal domain-driven conditional prediction patterns

>* Limit interactions to avoid overfitting and complexity
>* Use pipelines, validation, and regularization



In [ ]:
#@title Python Code - Interaction Features

# Interaction features capture joint feature behavior.
# This example uses only basic Python.
# Small data keeps the output readable.

# No extra installations are required.

# Store tiny house examples in dictionaries.
homes = [
    {"size_sqft": 900, "distance_miles": 2.0, "price_k": 310},
    {"size_sqft": 1200, "distance_miles": 5.0, "price_k": 330},

    {"size_sqft": 1500, "distance_miles": 1.5, "price_k": 520},
    {"size_sqft": 1800, "distance_miles": 8.0, "price_k": 410},
]

# Validate the tiny dataset before expansion.
assert len(homes) >= 2 and all("size_sqft" in row for row in homes)

# Create interaction features from existing measurements.
expanded_rows = []
for row in homes:
    interaction = row["size_sqft"] * row["distance_miles"]

    expanded_rows.append({
        "size_sqft": row["size_sqft"],
        "distance_miles": row["distance_miles"],
        "size_x_distance": interaction,
        "price_k": row["price_k"],
    })

# Print a compact before and after comparison.
print("Original features: size_sqft, distance_miles")
print("Expanded feature: size_sqft * distance_miles")

# Show only a few rows to avoid clutter.
for row in expanded_rows[:4]:
    message = (
        f"{row['size_sqft']} sqft, {row['distance_miles']} miles -> "
        f"interaction {row['size_x_distance']:.0f}, price ${row['price_k']}k"
    )

    print(message)

# Summarize why the new column can help.
print("Interaction features let simple models learn combined effects.")



### **2.3. Spline Feature Basis**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_B/image_02_03.jpg?v=1783906463" width="250">



>* Smooth local curves model continuous features
>* Knots capture changing nonlinear patterns

>* Split one feature into flexible curve parts
>* Knots balance smoothness and overfitting risk

>* Fit spline preprocessing on training data
>* Model smooth nonlinear changes without excess complexity



In [ ]:
#@title Python Code - Spline Feature Basis

# Build spline basis features manually.
# Compare smooth local feature activations.
# Keep preprocessing learned from training.

# Import numerical and plotting tools.
import numpy as np
import matplotlib.pyplot as plt

# Create deterministic one dimensional training data.
rng = np.random.default_rng(7)
x_train = np.linspace(0.0, 100.0, 80)

# Define knots using training data quantiles.
knots = np.quantile(x_train, [0.25, 0.50, 0.75])
assert knots.size == 3 and x_train.size == 80

# Build a simple cubic truncated spline basis.
def spline_basis(values, learned_knots):
    values = np.asarray(values, dtype=float)
    columns = [np.ones_like(values), values]

    # Add localized cubic bend features.
    for knot in learned_knots:
        columns.append(np.maximum(values - knot, 0.0) ** 3)

    # Stack columns into a design matrix.
    basis = np.column_stack(columns)
    return basis

# Transform training and future values consistently.
x_future = np.linspace(0.0, 100.0, 200)
train_basis = spline_basis(x_train, knots)
future_basis = spline_basis(x_future, knots)

# Validate the expanded feature counts.
assert train_basis.shape == (80, 5)
assert future_basis.shape == (200, 5)

# Print a compact summary for learners.
print(f"Training rows: {train_basis.shape[0]}")
print(f"Spline feature columns: {train_basis.shape[1]}")
print(f"Learned knots: {np.round(knots, 1).tolist()}")

# Plot only the localized spline features.
plt.figure(figsize=(8, 4))
for index, knot in enumerate(knots, start=2):
    plt.plot(x_future, future_basis[:, index], label=f"knot {knot:.1f}")

# Label the single teaching plot.
plt.title("Cubic spline basis features")
plt.xlabel("Temperature in degrees Fahrenheit")
plt.ylabel("Feature activation")
plt.legend()

# Display the plot in Colab.
plt.tight_layout()
plt.show()



## **3. Safe Pipeline Transforms**

### **3.1. Custom Function Transforms**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_B/image_03_01.jpg?v=1783906477" width="250">



>* Create domain-specific features with custom transformers
>* Apply them safely inside prediction-time pipelines

>* Make custom transforms consistent and explainable
>* Learn data rules only inside pipelines

>* Keep preprocessing consistent from training to deployment
>* Make custom features reproducible and leakage-safe



### **3.2. Reversing Transformations**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_B/image_03_02.jpg?v=1783906479" width="250">



>* Transformations change model inputs from original units
>* Inverse transforms make outputs understandable and usable

>* Fit reversible transforms on training data only
>* Reuse stored parameters to avoid leakage

>* Some transformations permanently lose original details
>* Document reversibility for safe, explainable pipelines



### **3.3. Prevent Data Leakage**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_04/Lecture_B/image_03_03.jpg?v=1783906481" width="250">



>* Leakage makes model scores look too optimistic
>* Pipelines fit transforms on training data only

>* Cross-validation folds need separate preprocessing
>* Pipelines fit transforms on training data only

>* Use training-learned rules for future data
>* Fit preprocessing inside the model pipeline



# <font color="#418FDE" size="6.5" uppercase>**Expansion Binning**</font>


In this lecture, you learned to:
- Use binarization and discretization to convert continuous features into thresholded or binned representations. 
- Generate polynomial, interaction, and spline features for nonlinear modeling. 
- Build leakage-safe preprocessing steps using transformers inside pipelines. 

In the next Module (Module 5), we will go over 'None'